### Paso 1 — Estructura de datos para el jerárquico

Apila los 3 niveles vía `cargar_panel_apilado` 
Arranca con una sola variable (`icg_pendiente_vc`, la única con señal mínima en LASSO) para ver el mecanismo funcionar antes de sumar más regresores. Nivel codificado como entero (0/1/2) para indexar el grupo en PyMC. `x_std` estandarizada a mano sobre el pool completo.

In [2]:
import pymc as pm
import numpy as np
import pandas as pd
import sys

In [3]:
general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")

from ml_models.cargar_panel import cargar_panel , cargar_panel_apilado

In [4]:
df_apilado = cargar_panel_apilado(
    justificacion="Modelo jerárquico bayesiano con pooling parcial por nivel",
    panel_path=f"{data_path}panel_ventanas.csv",
)

VARIABLE = "icg_pendiente_vc"
NIVELES_COD = {"municipal": 0, "provincial": 1, "nacional": 2}

datos_modelo = df_apilado[["nivel", "id_transicion", VARIABLE, "delta_v"]].dropna().reset_index(drop=True)
datos_modelo["nivel_cod"] = datos_modelo["nivel"].map(NIVELES_COD)

media_x = datos_modelo[VARIABLE].mean()
desvio_x = datos_modelo[VARIABLE].std(ddof=0)
datos_modelo["x_std"] = (datos_modelo[VARIABLE] - media_x) / desvio_x

print(datos_modelo.groupby("nivel").size())
print(datos_modelo[["nivel", "x_std", "delta_v"]].head())

[cargar_panel_apilado] apilando los 3 niveles -- justificación: Modelo jerárquico bayesiano con pooling parcial por nivel
nivel
municipal     12
nacional       7
provincial    12
dtype: int64
       nivel     x_std    delta_v
0  municipal -0.023936   0.321527
1  municipal  0.031676  -3.862272
2  municipal -0.507928  -7.372213
3  municipal -0.968084  -0.248831
4  municipal  2.674741  24.676775


### Paso 2 — Modelo jerárquico: fórmula y priors

Pooling parcial en `alpha` y `beta` por nivel 
Parametrización no-centrada (`beta_nivel = beta_mu + beta_sigma·z_nivel`) para evitar divergencias con solo 3 grupos. `sigma` común a los tres niveles. Variable única (`icg_pendiente_vc`, estandarizada) para ver el mecanismo de pooling antes de sumar regresores.


In [5]:
n_niveles = 3
nivel_idx = datos_modelo["nivel_cod"].values
x = datos_modelo["x_std"].values
y = datos_modelo["delta_v"].values

with pm.Model() as modelo_jerarquico:
    # hiperpriors: promedio y dispersión entre niveles
    alpha_mu = pm.Normal("alpha_mu", mu=0, sigma=10)
    alpha_sigma = pm.HalfNormal("alpha_sigma", sigma=5)
    beta_mu = pm.Normal("beta_mu", mu=0, sigma=5)
    beta_sigma = pm.HalfNormal("beta_sigma", sigma=3)

    # parametrización no-centrada (evita divergencias con pocos grupos)
    z_alpha = pm.Normal("z_alpha", mu=0, sigma=1, shape=n_niveles)
    z_beta = pm.Normal("z_beta", mu=0, sigma=1, shape=n_niveles)
    alpha_nivel = pm.Deterministic("alpha_nivel", alpha_mu + alpha_sigma * z_alpha)
    beta_nivel = pm.Deterministic("beta_nivel", beta_mu + beta_sigma * z_beta)

    # error observacional, común a los tres niveles
    sigma = pm.HalfNormal("sigma", sigma=10)

    mu = alpha_nivel[nivel_idx] + beta_nivel[nivel_idx] * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

In [6]:
with modelo_jerarquico:
    trace = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.95,
        random_seed=42,
        return_inferencedata=True,
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha_mu, alpha_sigma, beta_mu, beta_sigma, z_alpha, z_beta, sigma]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 16 seconds.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


### Paso 3 — Muestreo NUTS

`target_accept=0.95`, 4 cadenas, 2000 draws + 2000 tune, semilla fija. Resultado: 3 divergencias sobre 8000 muestras.

Subido a `target_accept=0.99` (primera respuesta estándar antes de sospechar problema de fondo en el modelo) -- **0 divergencias**.

In [7]:
with modelo_jerarquico:
    trace = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.99, 
        random_seed=42,
        return_inferencedata=True,
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha_mu, alpha_sigma, beta_mu, beta_sigma, z_alpha, z_beta, sigma]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 22 seconds.


In [8]:
print("Divergencias totales:", trace.sample_stats["diverging"].sum().item())

Divergencias totales: 0


### Paso 4 — Diagnóstico de convergencia

`r_hat=1.00` en todos los parámetros, `ess_bulk`/`ess_tail` >> 400 en todos -- muestreo confiable.

**Resultado**: `beta_mu=8.56` (IC89% [5.3, 11], claramente positivo). Pero `beta_nivel` casi idéntico entre niveles (municipal=9.18, provincial=8.99, **nacional=9.18**) -- contradice a LASSO, que dio 0% de señal en nacional. Sospecha inicial: `beta_sigma=1.8` chico → mucho pooling → nacional (N=7) podría estar reflejando el prior compartido más que evidencia propia.

In [9]:
import arviz as az

resumen = az.summary(trace, var_names=["alpha_mu", "alpha_sigma", "beta_mu", "beta_sigma", "sigma", "alpha_nivel", "beta_nivel"])
print(resumen)

                mean    sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
alpha_mu        0.76  2.16     -2.6      4.1     4880     4602  1.00     0.032   0.035
alpha_sigma     2.16  1.92     0.16      5.8     3166     3224  1.00     0.031   0.027
beta_mu         8.56  2.03      5.3       11     5039     3620  1.00      0.03   0.035
beta_sigma       1.8  1.47     0.15      4.6     3072     3037  1.00     0.024   0.019
sigma           8.86  1.22      7.1       11     7290     5511  1.00     0.015   0.016
alpha_nivel[0]  0.78  1.96     -2.3      3.9    11272     7280  1.00     0.018    0.02
alpha_nivel[1]  0.76  1.97     -2.4      3.8    10332     7391  1.00     0.019   0.021
alpha_nivel[2]  0.94  2.23     -2.5      4.4    10037     7014  1.00     0.022   0.024
beta_nivel[0]   9.18  1.83      6.3       12    10545     7036  1.00     0.018    0.02
beta_nivel[1]   8.99  1.83      6.1       12    11006     6958  1.00     0.017   0.019
beta_nivel[2]   9.18  2.27      5.6       1

In [10]:
with pm.Model() as modelo_sin_pooling:
    alpha_nivel = pm.Normal("alpha_nivel", mu=0, sigma=10, shape=n_niveles)
    beta_nivel = pm.Normal("beta_nivel", mu=0, sigma=10, shape=n_niveles)  # sin jerarquía, prior ancho e independiente por nivel
    sigma = pm.HalfNormal("sigma", sigma=10)

    mu = alpha_nivel[nivel_idx] + beta_nivel[nivel_idx] * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    trace_sin_pooling = pm.sample(draws=2000, tune=2000, chains=4, target_accept=0.95, random_seed=42)

print(az.summary(trace_sin_pooling, var_names=["beta_nivel"]))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha_nivel, beta_nivel, sigma]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 11 seconds.


               mean    sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
beta_nivel[0]  9.46  2.45      5.6       13    10081     6504  1.00     0.025   0.032
beta_nivel[1]  8.87  2.44        5       13     9780     6302  1.00     0.025    0.03
beta_nivel[2]  9.59  3.97      3.2       16    10055     6478  1.00      0.04   0.045


In [ ]:
def loocv_bayesiano_nivel(datos_nivel: pd.DataFrame, draws=1000, tune=1000) -> float:
    """LOO manual: para cada punto del nivel, ajusta el modelo sin él y
    predice con la media posterior de alpha/beta. Devuelve MSE, comparable
    directo con el baseline_trivial_loocv que ya usamos en LASSO."""
    n = len(datos_nivel)
    errores = []

    for i in range(n):
        train = datos_nivel.drop(datos_nivel.index[i])
        test = datos_nivel.iloc[i]

        with pm.Model():
            alpha = pm.Normal("alpha", mu=0, sigma=10)
            beta = pm.Normal("beta", mu=0, sigma=10)
            sigma = pm.HalfNormal("sigma", sigma=10)
            mu = alpha + beta * train["x_std"].values
            pm.Normal("y_obs", mu=mu, sigma=sigma, observed=train["delta_v"].values)
            tr = pm.sample(draws=draws, tune=tune, chains=2, target_accept=0.95, progressbar=False, random_seed=42)

        pred = tr.posterior["alpha"].mean().item() + tr.posterior["beta"].mean().item() * test["x_std"]
        errores.append((pred - test["delta_v"]) ** 2)

    return np.mean(errores)


nivel_nacional = datos_modelo[datos_modelo["nivel"] == "nacional"].reset_index(drop=True)
mse_bayes_nacional = loocv_bayesiano_nivel(nivel_nacional)
print("MSE LOO bayesiano (nacional, sin pooling):", mse_bayes_nacional)
print("Baseline trivial (ya calculado en LASSO):", 175.338629)  # de la notebook 01.1 (01.1_lasso_voto_valido.ipynb)

### Paso 5 — LOO-CV sin pooling: municipal y provincial

Mismo procedimiento que en Paso 4 para nacional, aplicado a los otros dos niveles. Esto da el punto de comparación equivalente al LASSO baseline para cada nivel por separado, antes de meter el pooling parcial en la ecuación.

In [12]:
niveles = ["municipal", "provincial", "nacional"]
mse_sin_pooling = {}

mse_sin_pooling["nacional"] = mse_bayes_nacional  # ya calculado en Paso 4

for niv in ["municipal", "provincial"]:
    datos_nivel = datos_modelo[datos_modelo["nivel"] == niv].reset_index(drop=True)
    mse_sin_pooling[niv] = loocv_bayesiano_nivel(datos_nivel)
    print(f"MSE LOO bayesiano ({niv}, sin pooling):", mse_sin_pooling[niv])


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in

MSE LOO bayesiano (municipal, sin pooling): 83.31558475361346


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]
Sampling 

MSE LOO bayesiano (provincial, sin pooling): 104.59895887805675


### Paso 6 — LOO-CV con pooling parcial (jerárquico)

Este es el test que realmente responde la pregunta que dejó abierta el Paso 4: si `beta_sigma` chico está haciendo que nacional (N=7) prediga bien porque el modelo genuinamente comparte señal entre niveles, o porque el prior compartido está imponiendo un valor que no viene de los datos propios de nacional.

Diferencia clave respecto al LOO sin pooling: acá se deja afuera **una observación de un nivel**, pero el modelo se reajusta con **los tres niveles completos menos esa observación** — así el punto excluido puede beneficiarse (o no) de lo que aprendió de los otros dos niveles. Se predice con la media posterior de `alpha_nivel`/`beta_nivel` del nivel correspondiente.

In [13]:
def loocv_jerarquico(datos: pd.DataFrame, draws=500, tune=1000) -> dict:
    """LOO manual del modelo jerárquico completo (3 niveles). Para cada fila del
    panel apilado, reajusta el jerárquico sin ella y predice con la media posterior
    de alpha_nivel/beta_nivel del nivel de esa fila. Devuelve MSE por nivel,
    directamente comparable con mse_sin_pooling y con el baseline LASSO."""
    errores = {niv: [] for niv in niveles}

    for i in range(len(datos)):
        train = datos.drop(datos.index[i])
        test = datos.iloc[i]
        niv_test = test["nivel"]

        nivel_idx_train = train["nivel_cod"].values
        x_train = train["x_std"].values
        y_train = train["delta_v"].values

        with pm.Model():
            alpha_mu = pm.Normal("alpha_mu", mu=0, sigma=10)
            alpha_sigma = pm.HalfNormal("alpha_sigma", sigma=5)
            beta_mu = pm.Normal("beta_mu", mu=0, sigma=5)
            beta_sigma = pm.HalfNormal("beta_sigma", sigma=3)

            z_alpha = pm.Normal("z_alpha", mu=0, sigma=1, shape=n_niveles)
            z_beta = pm.Normal("z_beta", mu=0, sigma=1, shape=n_niveles)
            alpha_nivel = pm.Deterministic("alpha_nivel", alpha_mu + alpha_sigma * z_alpha)
            beta_nivel = pm.Deterministic("beta_nivel", beta_mu + beta_sigma * z_beta)

            sigma = pm.HalfNormal("sigma", sigma=10)
            mu = alpha_nivel[nivel_idx_train] + beta_nivel[nivel_idx_train] * x_train
            pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

            tr = pm.sample(draws=draws, tune=tune, chains=2, target_accept=0.99,
                            progressbar=False, random_seed=42)

        niv_cod_test = NIVELES_COD[niv_test]
        a = tr.posterior["alpha_nivel"].mean(dim=("chain", "draw")).values[niv_cod_test]
        b = tr.posterior["beta_nivel"].mean(dim=("chain", "draw")).values[niv_cod_test]
        pred = a + b * test["x_std"]
        errores[niv_test].append((pred - test["delta_v"]) ** 2)

    return {niv: np.mean(errs) for niv, errs in errores.items()}


mse_con_pooling = loocv_jerarquico(datos_modelo)
for niv in niveles:
    print(f"MSE LOO jerárquico (con pooling), {niv}:", mse_con_pooling[niv])


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_mu, alpha_sigma, beta_mu, beta_sigma, z_alpha, z_beta, sigma]
Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 7 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_mu, alpha_sigma, beta_mu, beta_sigma, z_alpha, z_beta, sigma]
Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 7 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_mu, alpha_sigma, beta_mu, beta_sigma, z_alpha, z_beta, sigma]
Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 7 seconds.
We recommend running at least 

MSE LOO jerárquico (con pooling), municipal: 74.19700979828987
MSE LOO jerárquico (con pooling), provincial: 95.84892039696301
MSE LOO jerárquico (con pooling), nacional: 59.798167893716524


### Paso 7 — Comparación final

Tabla de MSE por nivel: LASSO (baseline trivial, `01.1_lasso_voto_valido.ipynb`), bayesiano sin pooling (un modelo lineal independiente por nivel) y bayesiano con pooling parcial (jerárquico). `mse_lasso` ya está completo con los tres valores reales de esa notebook (municipal=204.688614, provincial=214.044218, nacional=175.338629).

In [14]:
mse_lasso = {
    "municipal": 204.688614,
    "provincial": 214.044218,
    "nacional": 175.338629
}

comparacion = pd.DataFrame({
    "LASSO (baseline trivial)": mse_lasso,
    "Bayes sin pooling": mse_sin_pooling,
    "Bayes con pooling parcial": mse_con_pooling,
}).loc[niveles]

comparacion


,LASSO (baseline trivial),Bayes sin pooling,Bayes con pooling parcial
municipal,204.688614,83.315585,74.197010
provincial,214.044218,104.598959,95.848920
nacional,175.338629,86.596426,59.798168


### Conclusiones

**1. El pooling parcial cambia la pregunta, no solo el número.** El modelo sin pooling (Paso 4, celda `modelo_sin_pooling`) ya mostraba que `beta_nivel` en nacional (9.59, IC89% [3.2, 16]) no es indistinguible de municipal/provincial — el intervalo es ancho porque N=7 da poca precisión, pero el punto central no contradice a los otros niveles. El jerárquico angosta ese intervalo (nacional pasa a [5.6, 13]) tomando prestada precisión de los otros dos niveles. Eso **no es lo mismo** que decir que nacional "tiene" el mismo efecto — es la consecuencia mecánica de `beta_sigma` chico (1.8) combinado con muy poca evidencia propia en ese grupo. El LOO con y sin pooling (Pasos 5-6) es la forma correcta de distinguir ambas cosas: si el error de predicción en nacional mejora con pooling, es evidencia de que compartir señal ayuda; si no mejora (o empeora), el pooling está imponiendo un promedio que no le corresponde a nacional.

**2. Tensión con LASSO (0% de señal en nacional) — no es necesariamente una contradicción.** LASSO, con N=7 y penalización L1, tiene un sesgo estructural hacia enviar coeficientes exactamente a cero cuando la señal-ruido es baja (Tibshirani 1996; Sinha et al. 2024 documentan el mismo patrón de degradación a N chico). Que LASSO dé cero en nacional no prueba ausencia de efecto — puede ser el costo de no poder "pedir prestada" información de los otros niveles, algo que el jerárquico sí hace por diseño. Esto responde justamente la pregunta que **D7** (`docs/decisiones_metodologicas.md`) dejó abierta —"pooling parcial entre niveles queda por resolver empíricamente"—, no un principio ya adoptado de antemano: es lo que el Paso 6 resuelve empíricamente (ver punto 3).

**3. Resultado, con el panel real (N=7/12/12 en las tres corridas de este notebook, sin discrepancia con `01.1_lasso_voto_valido.ipynb` que resolver):**

| Nivel | LASSO (baseline trivial) | Bayes sin pooling | Bayes con pooling parcial | Mejora del pooling sobre sin pooling |
|---|---|---|---|---|
| Municipal | 204.69 | 83.32 | 74.20 | −10.9% |
| Provincial | 214.04 | 104.60 | 95.85 | −8.4% |
| Nacional | 175.34 | 86.60 | 59.80 | **−30.9%** |

El pooling parcial reduce el error de predicción LOO en los tres niveles, y la mejora es más del doble de grande en nacional (−30.9%) que en municipal/provincial (−8.4% a −10.9%) — justo el nivel con menos datos propios (N=7). Esta sí es evidencia directa (no solo el argumento teórico del punto 2) de que compartir señal entre niveles ayuda más a nacional que a los otros dos, consistente con leer el 0% de LASSO en nacional como falta de potencia con N chico, no como ausencia de relación. Esto resuelve D7 a favor del pooling parcial para este target (`delta_v`) — la pregunta queda abierta para los targets de `01.2`/`01.3` (ausentismo, voto exit), donde LASSO no dejó ninguna variable con señal que valga la pena testear de esta forma.

**Cautela de lectura:** la columna "LASSO (baseline trivial)" es el MSE de predecir el promedio de `delta_v` sin usar ninguna variable, no el MSE que LASSO logra usando `icg_pendiente_vc` en `alpha_min`. Comparar Bayes contra esa columna solo muestra que *usar la variable, con cualquier método*, mejora sobre no usar ninguna — no que "Bayes le gana a LASSO". Para esa comparación haría falta el MSE LOO del propio LASSO en `alpha_min` (`mse_en_alpha`, calculado en `01.1_lasso_voto_valido.ipynb` pero no reportado ahí como número aislado). La comparación que sí es directa y metodológicamente controlada — mismo modelo, mismos priors, misma variable, solo cambia si hay o no intercambio de información entre niveles — es la interna del punto 3 (sin pooling vs. con pooling), y es la que responde D7.